In [1]:
from importlib.metadata import version

# Check versions of key packages
pkgs = ["matplotlib",
        "scikit-learn",
        "numpy",
        "pandas",
        "torch",
        "langchain",
        "langgraph",
        "faiss-cpu"
       ]

# iterate and print versions
for p in pkgs:
    print(f"{p} version: {version(p)}")

import sys
print(f"Python version: {sys.version}")

matplotlib version: 3.10.6
scikit-learn version: 1.7.2
numpy version: 2.2.6
pandas version: 2.3.2
torch version: 2.8.0
langchain version: 0.3.27
langgraph version: 1.0.1
faiss-cpu version: 1.12.0
Python version: 3.13.7 | packaged by Anaconda, Inc. | (main, Sep  9 2025, 19:54:17) [Clang 17.0.6 ]


In [2]:
# 1. Load and Chunk PDF Documents
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- Configuration ---
LOCAL_MODEL_NAME = "/Users/sir/Downloads/HuggingFace/sentence_transformer/intfloat_e5-large-v2" # Used for Embeddings
LLM_MODEL_NAME = "/Users/sir/Downloads/HuggingFace/LLM/Mistral-7B-Instruct-v0.3" # Placeholder for Local LLM
RERANKER_MODEL_NAME = "/Users/sir/Downloads/HuggingFace/cross_encoder/BAAI_bge-reranker-large" # Cross-encoder model
FAISS_INITIAL_K = 10 # Number of documents to fetch from FAISS (input to Reranker)ls
RERANKER_FINAL_K = 5 # Number of documents to pass to the LLM (output from Reranker)
SAVE_DIR = "/Users/sir/Downloads/HuggingFace/VectorDB/faiss_local_index"
PDF_PATH = "/Users/sir/Desktop/Project/Data/NLP/Book/LLMs-in-Production.pdf" 
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

In [3]:
# --- 1. Load and Chunk PDF Documents ---
print("--- 1. Loading and Chunking Documents ---")
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
documents = text_splitter.split_documents(docs)
print(f"Loaded and split into {len(documents)} chunks.")

Overwriting cache for 0 3229


--- 1. Loading and Chunking Documents ---
Loaded and split into 2545 chunks.


In [4]:
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# --- 2. Embed and Store in FAISS ---
print("\n--- 2. Setting up FAISS and Embeddings ---")
embedding_model = HuggingFaceEmbeddings(model_name=LOCAL_MODEL_NAME)

if not os.path.exists(SAVE_DIR):
    print("Creating new FAISS vector store...")
    vectorstore = FAISS.from_documents(documents, embedding_model)
    vectorstore.save_local(SAVE_DIR)

vectorstore = FAISS.load_local(SAVE_DIR, embedding_model, allow_dangerous_deserialization=True)
print("FAISS vector store loaded.")


--- 2. Setting up FAISS and Embeddings ---
FAISS vector store loaded.


In [5]:
# --- 3. Create Base Retriever ---
retriever = vectorstore.as_retriever(search_kwargs={"k": FAISS_INITIAL_K})

In [6]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

# --- 4. Add Reranker ---
print("\n--- 4. Initializing Reranker ---")
reranker_model = HuggingFaceCrossEncoder(model_name=RERANKER_MODEL_NAME)
reranker = CrossEncoderReranker(model=reranker_model, top_n=RERANKER_FINAL_K)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=retriever
)
print("Compression Retriever (Reranker) initialized.")


--- 4. Initializing Reranker ---
Compression Retriever (Reranker) initialized.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

# --- 5. Load LLM (Mistral) ---
print(f"\n--- 5. Loading LLM: {LLM_MODEL_NAME} ---")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token 
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, dtype=torch.float16, device_map="auto")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)
llm = HuggingFacePipeline(pipeline=pipe)
print("LLM pipeline initialized.")


--- 5. Loading LLM: /Users/sir/Downloads/HuggingFace/LLM/Mistral-7B-Instruct-v0.3 ---


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Device set to use mps


LLM pipeline initialized.


In [20]:
from langchain_core.prompts import ChatPromptTemplate

# --- 6. RAG Prompt Template ---
# rag_prompt = ChatPromptTemplate.from_messages([
#     ("system", "Answer the question using the context below. If the answer is not in the context, state that you cannot find the answer."),
#     ("human", "Question: {input}\n\nContext:\n{context}")
# ])

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful expert. Answer the question STRICTLY using the context provided below. "
     "Do not use external knowledge or generate code examples. "
     "If the answer is not in the context, you MUST state that you cannot find the answer. "
     "Your response must be a direct answer to the question."
    ),
    ("human", "Question: {input}\n\nContext:\n{context}")
])


In [21]:
from langchain.chains.combine_documents import create_stuff_documents_chain

# --- 7. Document Chain ---
document_chain = create_stuff_documents_chain(llm, rag_prompt)

In [22]:
from langchain.chains.combine_documents import create_stuff_documents_chain

# --- 7. Document Chain  ---
document_chain = create_stuff_documents_chain(llm, rag_prompt)

In [23]:
from langchain_core.prompts import ChatPromptTemplate

# --- 8. History-Aware Retriever Prompt ---
history_aware_prompt = ChatPromptTemplate.from_messages([
    ("placeholder", "{chat_history}"),
    ("user", "{input}"),
    ("user", "Given the above conversation, generate a concise, standalone search query for the latest user question. Do not include any conversational filler."),
])

In [ ]:
from langchain.chains.history_aware_retriever import create_history_aware_retriever

# --- 9. History-Aware Retriever ---
history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=compression_retriever,
    prompt=history_aware_prompt
)
print("History-Aware Retriever initialized.")

History-Aware Retriever initialized.


In [25]:
from langchain.chains.retrieval import create_retrieval_chain

# --- 10. Define retrieval_chain ---
retrieval_chain = create_retrieval_chain(history_aware_retriever, document_chain)

In [29]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough

# --- 11. Final Conversational Chain Setup  ---
print("\n--- 11. Setting up Conversational Chain ---")

# 1. Simplified Answer Cleaning Function (Returns only the string to be assigned to 'output')
def clean_answer_string(output_dict):
    """
    Heuristically cleans the verbose Mistral output and returns only the answer string.
    This resolves the verbose output issue.
    """
    raw_answer = output_dict['answer'].strip()
    
    # Look for the final generated answer tag (your model appears to add "Answer: ")
    if "Answer: " in raw_answer:
        clean_answer = raw_answer.split("Answer: ", 1)[-1].strip()
        
    # Fallback cleaning for older, more verbose structures
    elif "Context:" in raw_answer:
        clean_answer = raw_answer.split("Context:", 1)[-1].strip()
    else:
        clean_answer = raw_answer
        
    # Final cleanup of any residual human prompt elements
    if "Human: Question:" in clean_answer:
        clean_answer = clean_answer.split("Human: Question:", 1)[0].strip()
        
    return clean_answer # Returns only the clean string!

# 2. History Factory
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Returns a new in-memory history store for a given session ID."""
    return InMemoryChatMessageHistory(session_id=session_id)


# 3. Create the final runnable with the output mapper attached
# This ADDS a new key 'output' whose value is the cleaned string, resolving KeyError.
final_runnable = retrieval_chain | RunnablePassthrough.assign(
    output=clean_answer_string 
)

# 4. Manually wrap the final_runnable with RunnableWithMessageHistory
final_rag_chain = RunnableWithMessageHistory(
    final_runnable,
    get_session_history,
    input_messages_key="input", 
    history_messages_key="chat_history",
    session_history_key="configurable", 
)

print("Final Conversational RAG Chain ready.")


--- 11. Setting up Conversational Chain ---
Final Conversational RAG Chain ready.


In [27]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
# --- 11. Conversational Chain Setup ---

# 1. Define the factory function (No change here)
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Returns a new history store for a given session ID."""
    return InMemoryChatMessageHistory(session_id=session_id)


# 2. Manually wrap the retrieval_chain with RunnableWithMessageHistory
# This explicitly turns your combined RAG chain into a conversational chain.
final_rag_chain = RunnableWithMessageHistory(
    retrieval_chain,
    get_session_history,
    # Define how to find the session ID in the input config
    input_messages_key="input", 
    history_messages_key="chat_history",
    # Define the key in the config where the session ID will be found
    # This must match what you pass in the config during invocation.
    # The 'configurable' key is mandatory for session management.
    session_history_key="configurable", 
)

print("Final Conversational RAG Chain ready.")


Final Conversational RAG Chain ready.


In [30]:
# --- Execution Example ---
session_id = "user_session_1" 

# 1. First question
question_1 = "What is the primary benefit of using LLMs in a production environment?"
print("\n" + "="*50)
print(f"Session ID: {session_id}")
print(f"--- Turn 1: {question_1} ---")

response_1 = final_rag_chain.invoke(
    {"input": question_1}, 
    config={"configurable": {"session_id": session_id}}
)
# We access the new, safe key 'output'
print(f"A: {response_1['output'].strip()}") 


# 2. Second, context-dependent question
question_2 = "What are the common risks associated with that?" 
print(f"\n--- Turn 2: {question_2} ---")

response_2 = final_rag_chain.invoke(
    {"input": question_2}, 
    config={"configurable": {"session_id": session_id}} 
)
# We access the new, safe key 'output'
print(f"A: {response_2['output'].strip()}")
print("="*50)



Session ID: user_session_1
--- Turn 1: What is the primary benefit of using LLMs in a production environment? ---
A: LLMs in Production

“When you hit the endpoint, you will get 503 errors; sometimes you get a text
response as if the model was generating text, but I think that’s a bug.” Serving an LLM
in a production environment—trying to meet the needs of so many clients—is no easy
feat. However, deploying a model that’s integrated into your system allows you more
control of the process, affording higher availability and maintainability than you can

application lifecycle, data pipeline, compute cost, security, 
and more. Get it wrong, and you may have a costly failure 
on your hands.
LLMs in Production  teaches you how to develop an LLMOps 
plan that can take an AI app smoothly from design to delivery. 
You’ll learn techniques for preparing an 
LLM dataset, cost-
effi  cient training hacks like LORA and RLHF, and industry 
benchmarks for model evaluation. Along the way, you’ll put



In [19]:
# 3. Third, context-dependent question
question_3 = "What is LLM?"
print(f"\n--- Turn 3: {question_3} ---")

response_3 = final_rag_chain.invoke(
    {"input": question_3},
    config={"configurable": {"session_id": session_id}}
)
# We access the new, safe key 'output'
print(f"A: {response_3['output'].strip()}")
print("="*50)



--- Turn 3: What is LLM? ---
A: LLM. Anyone can learn effective strategies by simply playing with models or from
purely online resources. In other words, it’s hard to believe that there is any real engi-
neering going on when the majority of players are simply using the “guess and check”
method. But this logic highlights a basic misunderstanding of what engineering is.
There’s a big difference between getting a model to solve your problem once and get-
ting it to solve every user’s problem every single time.

to build
Easier
to build
Shopping assistant
Customer service bot
Chatbot
Writing assistant
Text to image
Coding assistant
Explaining jokes
Explaining ethics
ChessBot
Q&A bot
Figure 2.9 How difficult or easy certain tasks are for LLMs and what approaches to take to solve them

better yet, “What shouldn’t they do?” 
 Well, as a technology, there are certain restrictions and constraints. For example,
LLMs are kind of slow. Of course, slow is a relative term, but responsive times are